In [3]:
import numpy as np
import pandas as pd

books = pd.read_csv('books_with_categories.csv')

## Sentiment Analysis:

We want to understand the `emotion` associated with each book and to get those emotions we're gonaa use a fine-tuned Huggingface model called **emotion-english-distilroberta-base**. The accurace of the model is 66\% which is significantly higher than random guess of 14\%.

 We want to classify books to different categories such as:
- Anger
- Joy
- Surprise
- Sadness
- Disgust
- Fear
- Neutral 

In [15]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("text-classification", model="j-hartmann/emotion-english-distilroberta-base", top_k=None)

Device set to use cpu


In [69]:
# Emotion categories
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

# function to extract max emotion score

def max_emotion_score(predictions):
    # to store all the sentences' emotion scores for each emotion label 
    per_emotion_scores = {label:[] for label in emotion_labels}
    for sentence_pred in predictions:
        sorted_pred = sorted(sentence_pred, key = lambda x : x['label'])
        for idx, label in enumerate(emotion_labels):
            #append the score to the score list for each label in emotion labels
            per_emotion_scores[label].append(sorted_pred[idx]['score'])
    return {label : np.max(scores) for label, scores in per_emotion_scores.items()}


In [72]:
# to store the max emotion score for each label and for each book
emotion_scores = {label: [] for label in emotion_labels}
isbn = []

for i in tqdm(range(len(books))):
    isbn.append(books['isbn13'][i])
    sentences = [s.strip() for s in books['description'][i].split('.') if s.strip()]
    predictions = pipe(sentences)
    max_score = max_emotion_score(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_score[label])

100%|██████████| 5197/5197 [05:52<00:00, 14.75it/s]


In [73]:
emotion_df = pd.DataFrame(emotion_scores)
emotion_df['isbn13'] = isbn

In [74]:
emotion_df

,anger,disgust,fear,joy,sadness,surprise,neutral,isbn13
0,0.029641,0.338239,0.983973,0.949028,0.697845,0.956065,0.729602,9780002005883
1,0.594469,0.461990,0.935215,0.704422,0.891109,0.051414,0.212222,9780002261982
2,0.041301,0.024568,0.973285,0.767237,0.042176,0.010860,0.009796,9780006178736
3,0.325305,0.125763,0.436339,0.242209,0.732685,0.043272,0.029084,9780006280897
4,0.091732,0.197434,0.095043,0.041146,0.890048,0.475881,0.074878,9780006280934
...,...,...,...,...,...,...,...,...
5192,0.204142,0.053939,0.923713,0.303278,0.807057,0.974266,0.028640,9788172235222
5193,0.058450,0.127941,0.025688,0.400263,0.891073,0.016014,0.227765,9788173031014
5194,0.011246,0.010868,0.314812,0.942169,0.344737,0.060437,0.056819,9788179921623
5195,0.034505,0.080506,0.409873,0.776054,0.950763,0.317455,0.049227,9788185300535


In [75]:
books = pd.merge(books, emotion_df, on = 'isbn13', how = 'left')

In [76]:
books.to_csv('books_with_emotions.csv', index=False)

In [77]:
!pwd

/Users/sahelkeyvan/Documents/Python projects/book_recommendation/.venv
